# 3.1 資料與時點處理

## 資料範圍與時點控制

**母體**　S&P 500 成分股，Tiingo 日頻價格，2000-01 至 2025-12，共 **6,287 個交易日**
**基本面**　SEC EDGAR XBRL companyfacts　**產業**　GICS 十一大類

### 三處時點（point-in-time）控制

| 環節 | 控制方式 | 若不控制的後果 |
| :--- | :--- | :--- |
| **成分股認定** | 依 `index_memberships` 的納入／剔除日，每期僅取**當時真實在指數內**者；含 170 檔已下市股 | 存活者偏誤 |
| **基本面對齊** | 一律取 `filed ≤ 形成期結束日` 的最新一筆，**非**財報期末日 | 使用尚未公開的資訊 |
| **特徵計算窗** | 僅用形成期窗內資料，交易期價格不參與任何估計 | 前視偏誤 |

資料庫共 **843 檔**成分股。

## 滾動回測設計

| 參數 | 設定 |
| :--- | :---: |
| 形成期長度 | **252** 個交易日（約一年） |
| 交易期長度 | **126** 個交易日（約半年） |
| 滾動步長 | **21** 個交易日（約一月） |
| 同時重疊期數 | **6**（= 126 / 21） |

::: {.callout-important}

### 一項對統計設計有決定性影響的性質

滾動步長 **小於** 交易期長度 → 任一時點有 **6 個交易期同時運行**
→ **逐期報酬序列存在結構性自相關**。

此性質決定了 §3.5 為何不能以「期」為抽樣單位，
以及為何必須使用 HAC 標準誤。

:::

**交易成本**　單邊 **0.29%**（Do & Faff, 2012 之 ~30bps 估計），
進出場各扣一次 → 一往返約名目額 **0.58%**。
該估計樣本期為 1962–2009，套用於 2000–2025 **偏保守**；
第四章另報 break-even 成本。

# 3.2 形成期的四層架構

## 設計原理：讓單變因成為結構保證

配對交易的形成期通常被實作為**單一整體流程** →
更換任一環節時難以歸因其效果。

本研究拆解為四個**可獨立替換**的層：

```
特徵萃取  →  分組  →  群內排序  →  統計篩選
```

各層以標準化介面銜接：
特徵層輸出 $(N 	imes d)$ 矩陣 → 分組層輸出 $\{股票 	o 組標籤\}$
→ 排序層在組內選前 $N$ 組 → 篩選層對價差施加統計檢定。

::: {.callout-tip}

此架構使「單變因對照」成為**結構上的保證**，而非人為約定——
檢定分組方法時，其餘三層的參數完全相同。

:::

## 特徵層：19 維（連續 7 維）

| 區塊 | 維度 | 內容 | 依據 |
| :--- | :---: | :--- | :--- |
| 報酬主成分載荷 | **5** | 形成期日報酬 PCA 前 5 主成分載荷，以特徵值平方根加權 | Avellaneda & Lee (2010) |
| 公司基本面 | **2** | 對數市值、盈餘殖利率（1/PE） | — |
| GICS 產業 one-hot | **12** | 11 大產業 + 1 未知 | — |

各區塊**獨立標準化**後依權重拼接（避免 one-hot 欄位數稀釋連續特徵的距離量測）。

缺失值以**產業中位數**插補後 winsorize（1%/99%）。

> ⚠️ 第四章將指出：此插補方式構成一條**未被察覺的產業資訊管道**，
> 並引入全域中位數插補作為對照。

## 分組層：三種分群 + 對照組

| 方法 | 關鍵參數 | 群數決定方式 |
| :--- | :--- | :--- |
| **HDBSCAN** | `min_cluster_size`=5, `min_samples`=2 | 資料驅動，可標記噪音 |
| **Agglomerative** | average linkage，門檻取距離分布 **75 分位** | 由門檻決定 |
| **K-means** | $k$ = **同期 Agglomerative 的群數** | 對齊使量級可比 |
| **GICS（對照）** | — | 11 大產業，不跑分群 |

::: {.callout-note}

### K-means 的群數為何要對齊

K-means 需**預先指定**群數，若任意給定將使其與其他方法不可比。
本研究先跑一次 Agglomerative 取得資料驅動的群數再餵給 K-means，
確保比較聚焦於**分群機制**而非**粒度**。

:::

HDBSCAN 的噪音點（標籤 −1）與過小群（成員 < 5）併入「Unknown」，於排序層跳過。

## 排序層與篩選層

### 排序層：三種距離準則

| 準則 | 定義 |
| :--- | :--- |
| **SSD** | 正規化對數價格路徑的平方差總和（GGR, 2006） |
| **DTW** | 動態時間校正，Sakoe-Chiba 頻帶寬 15（許鈞翔, 2025） |
| **SSD-DTW-PCA** | 兩距離的主成分融合，取第一主成分 |

### 篩選層：三道檢定（任一未過即淘汰）

1. **ADF 共整合**　殘差 $p < 0.05$（Engle & Granger, 1987）
2. **OU 半衰期**　$HL = -\ln 2 / \lambda$，要求 $1 \le HL \le 42$ 日
3. **Hurst 指數**　$H < 0.5$（均值回歸傾向）

半衰期上限 42 日 = 交易期 126 日的 1/3，確保價差有足夠時間回歸。

::: {.callout-important}

流程為「**先按距離排序、再逐一檢定並填滿名額**」——
故**候選池不足時，系統會被迫接受距離更遠的配對**。
第四章將顯示此機制是理解本研究結果的關鍵。

:::

# 3.3 「動態分群」的界定

標題所稱**動態分群**＝分群模型於**每一形成期獨立重新配適**，
而非全樣本分群一次後固定。

| 性質 | 內容 |
| :--- | :--- |
| **逐期重估** | 每期以該期 252 日視窗重建特徵、重新配適；不保留跨期狀態 |
| **重估次數** | **295 期**（2000-01-03 → 2024-07-01），三種分群法各配適 295 次 |
| **群數隨資料變動** | HDBSCAN 由密度決定；Agglomerative 取當期距離 75 分位；K-means 對齊之 |
| **前視偏誤防範** | 全樣本分群會讓 2005 年的搜尋空間受 2020 年共變影響 |

## 動態性的實測①：配對層級不具鑑別力

| 分組方法 | 相鄰期配對重疊率 | 配對存續期數（中位） |
| :--- | ---: | ---: |
| Agglomerative | 9.8% | 1 |
| HDBSCAN | 9.4% | 1 |
| K-means | 6.4% | 1 |
| **GICS（靜態對照）** | **11.7%** | 1 |

::: {.callout-warning}

### 靜態的 GICS 週轉率反而最高

配對週轉主要由**排序層與篩選層**驅動——即使分組固定，
每期的距離排序與共整合檢定結果本就不同。
**此指標無法佐證分群層的動態性。**

:::

## 動態性的實測②：分群層級以 ARI 量測

重跑形成期前兩層取出群標籤，計算相鄰期在**共同標的**上的
**調整蘭德指數**（群編號無意義，故比對「兩兩是否同群」；ARI 已對隨機一致校正）。

| 分組方法 | 群數 | 相隔 1 期（21 日） | 相隔 6 期（126 日） |
| :--- | ---: | ---: | ---: |
| Agglomerative | 104 | **0.709** | 0.483 |
| HDBSCAN | 16 | **0.685** | 0.498 |
| K-means | 104 | **0.515** | 0.350 |
| **GICS（靜態）** | 11 | **1.000** | 1.000 |

- **結構確實逐期改變**：相隔 21 日已有三至五成分群關係改變
- **改變隨時間累積**：一個交易期走完，過半結構已不同
- **K-means 最不穩定**（隨機初始化；Agglomerative 的階層合併較不敏感）

> **範圍聲明**：確立分群**確實在變**，但**未**檢定這種改變是否**有益**——
> 本研究無「靜態分群」對照組。逐期重估是前視偏誤防範的必要設計，
> 而非受檢定的處理。

## 消融矩陣：4 分組 × 3 排序

## 4 分組 × 3 排序消融矩陣

|  | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（對照）** | ✓ | ✓ | ✓ |
| HDBSCAN | ✓ | ✓ | ✓ |
| Agglomerative | ✓ | ✓ | ✓ |
| K-means | ✓ | ✓ | ✓ |

固定特徵層、篩選層與交易端 → **唯一變因為分組方法**。
每格再展開 `top_n` × `stop_loss`（5 × 3 = 15 種配置）。

同一排序準則下的 ML 分群與 GICS 構成直接對照，共 **9 組**（3 分群 × 3 排序）。

### 三項受控消融（用於失敗歸因）

| 消融 | 參數 | 檢驗什麼 |
| :--- | :--- | :--- |
| **產業先驗強度** | `sector_onehot_weight` ∈ {1.0, 0} | one-hot 對跨產業配對的距離懲罰 |
| **統計篩選** | `filter_mode` ∈ {coint, none} | 篩選的貢獻，及其與分群的**交互作用** |
| **分組維度零點** | `cluster_method` = **none** | 「限制搜尋空間」本身的價值 |

> 若無「不分組」對照，所比較的僅是不同的**限制方式**，而非限制本身。

# 3.4 深度學習法（命題 2）

## 動作空間：門檻選擇式而非逐日定位

**基準**　Z-Score 規則：$|z| > 2.0$ 進場、$z$ 穿越 0 平倉、期末強平、另設停損。

> **動作選單（9 個）**：SKIP（不交易）
> ＋ 8 組 $(entry\_z,\ exit\_z) \in \{1.5, 2.0, 2.5, 3.0\} 	imes \{0.0, 0.5\}$

代理人對每組配對、每期**僅做一次決策**，選定後交由標準 Z-Score 狀態機執行整期。

### 三項結構性保證

1. 選單含靜態基準 $(2.0, 0.0)$ → **策略空間必然包含 Z-Score 基準**
2. 訓練樣本不足時自動選基準動作 → 樣本初期行為**等同**基準
3. **SKIP** 使代理人可拒絕交易 —— 固定規則不具備的選擇性

::: {.callout-note}

### 為何不用逐日定位的動作空間

本研究早期版本採「每日自由決定持倉」，結果模型對**日級噪音計時**，
換手成本大幅上升、績效顯著劣於基準。
失敗根因被隔離於**動作空間設計**，而非學習演算法本身。

:::

## 狀態空間與學習問題的性質

**狀態 = 12 維形成期特徵**（全部可於交易期開始前計算）

| 類別 | 特徵 |
| :--- | :--- |
| 偏離狀態 | 期末 $z$、期末 $\|z\|$ |
| 回歸品質 | 零穿越頻率、OU 半衰期對數 |
| 近期 regime | 近 21 日 $z$ 波動相對全期、近 21 日 $z$ 趨勢 |
| 配對性質 | 兩檔報酬相關係數、波動比、對沖比例偏離 1 的幅度 |
| 可交易性 | 價差振幅、形成期 $\|z\|>2$ 佔比、形成期最大 $\|z\|$ |

::: {.callout-tip}

### 這是全資訊監督回歸，不是 bandit

歷史配對期中，**全部 9 個動作的報酬都可精確反事實回算**
（對該期價格逐一模擬 9 組門檻）→ **無探索問題、樣本效率最高**。

網路：MLP（12 → 隱藏 64 → 9 個動作報酬），MSE 損失，每期增量訓練 40 epoch。

:::

## 前視偏誤的防範與隨機性處理

::: {.callout-important}

### walk-forward 增量訓練

> 期 $k$ 的決策，**僅使用「交易期已於期 $k$ 開始前結束」的樣本**訓練。

實作為對訓練緩衝區施加 `trade_end < trade_start_k` 過濾。
可用樣本 < 200 筆時，自動選用基準動作 $(2.0, 0.0)$。

:::

**隨機性處理**　網路未固定隨機種子 → 對三種 ML 配對底各執行**五輪獨立重訓**，
以中位數與全距報告，避免單次訓練的隨機性被誤讀為方法效果。

# 3.5 統計檢定方法

## 抽樣單位：為何不能用參數網格

本研究早期版本以「參數網格」為抽樣單位——對 15 種 `top_n` × `stop_loss`
配置作配對 $t$ 檢定。

::: {.callout-important}

### 偽重複（pseudo-replication）

15 個「觀測」共用**同一份資料、同一段期間、同一批配對**：

- `top_n`=10 與 `top_n`=20 **共用 10 組配對**
- 三種停損是**同一批交易**的不同出場規則

觀測間高度相關 → 不滿足 $t$ 檢定的獨立性假設 →
**有效樣本數接近一條回測路徑，而非 15**。

:::

**本研究改以「時間」為抽樣單位。**

## 主檢定與對照檢定

### 主檢定：逐日報酬差 + Newey-West HAC

$$\Delta r_t = r_{	ext{處理},t} - r_{	ext{對照},t}, \quad t = 1, \dots, 6287$$

檢定 $H_0: E[\Delta r] = 0$，Bartlett kernel HAC 標準誤（Newey & West, 1987），
吸收 6 期重疊部位造成的自相關。

**未持倉日記為 0**，非遺漏值——該日確實沒有部位；
視為遺漏將系統性排除策略「不交易」的行為。

**落後階同時報 {auto, 63, 126, 252}**——經驗法則在 $Tpprox6300$ 給出 10 階，
但持有期達 126 日，自相關可能延伸更遠。結論若隨落後階改變即為不穩健。

### 對照檢定：循環 block bootstrap

HAC 依賴常態近似，而配對交易日報酬**厚尾偏態**（大量零值日 + 少數大額平倉）。
去平均施加 $H_0$ → 首尾相接成環 → 抽長度 $L$ 區塊重組 → 5,000 次。

$L \in \{21, 126, 252\}$。**$L=126$ 為原則下限**；
若僅 $L=21$ 顯著，代表結論依賴切斷長程相關，不可採信。

## 多重檢定、絕對績效與非劣性

**多重檢定校正**　同一命題涉及多組對照時（如命題 1 的 9 組），
以 **Benjamini-Hochberg** 控制 FDR，同時報告原始與校正後 $p$ 值。

**絕對績效**
- Newey-West 絕對檢定：$H_0$ 為平均日報酬 = 0，**無對照組**
- **Deflated Sharpe Ratio**（Bailey & López de Prado, 2014）：
  校正「多次試驗中挑最佳者」的選擇偏誤。
  試驗數 $N$ 以資料庫**相異策略數 87** 為主口徑，
  附錄並列 $N \in \{15,\ 87,\ 1659\}$ —— $N$ 是判斷而非事實，故三者並陳。

::: {.callout-important}

### 非劣性檢定

「無顯著差異」**不等於**「兩者相當」。

主張後者需**預先指定**可容忍劣化 $\delta$，並證明單尾 95% 信賴下界 $> -\delta$。
本研究於命題 1 報告 $\delta \in \{0.25,\ 0.5,\ 1.0\}$ 個百分點的判定。

:::

## 本章小結

::: {.callout-important}

本研究的方法設計圍繞一項原則：

> **任兩個待比較的策略之間，僅存在單一變因。**

- **形成期**：四層架構使此原則成為**結構上的保證**
- **交易期**：利用「可在同一批配對上施行」的性質達成同樣效果

統計方面，以**時間**為抽樣單位取代參數網格，
並以 **HAC 與 block bootstrap 雙軌**檢定，
使相對比較的推論基礎不依賴單一方法論假設。

:::